In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Importamos librerias necesarias para knn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve


#Cargar el dataset
from google.colab import drive
drive.mount('/content/drive')
#ID del archivo
#https://drive.google.com/file/d/1wLoiFHQx7zuzdWlf0_8c7bHDmdt9qBbX/view?usp=sharing

file_id = "1wLoiFHQx7zuzdWlf0_8c7bHDmdt9qBbX"

url = f"https://drive.google.com/uc?id={file_id}"

#cargar datos
prestamos_df=pd.read_csv(url)
prestamos_df.head()

Mounted at /content/drive


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


In [5]:
# Assuming 'grade', 'purpose', 'addr_state', 'home_ownership' are categorical columns
# that need to be encoded into numerical representations with a '_code' suffix.

# List of categorical columns to encode
categorical_cols_to_encode = ['grade', 'purpose', 'addr_state', 'home_ownership']

for col in categorical_cols_to_encode:
    if col in prestamos_df.columns:
        prestamos_df[f'{col}_code'] = prestamos_df[col].factorize()[0]
    else:
        # Print a warning if the original categorical column is not found
        print(f"Warning: Categorical column '{col}' not found in prestamos_df. Skipping encoding for '{col}_code'.")

# Define the list of columns for X, including both original numerical and newly created encoded columns
# Initialize with known existing numerical columns
selected_features = ['funded_amnt', "int_rate", 'annual_inc', 'dti', 'revol_util', 'pub_rec_bankruptcies']

# Add the '_code' columns only if they were successfully created
for col in categorical_cols_to_encode:
    encoded_col_name = f'{col}_code'
    if encoded_col_name in prestamos_df.columns:
        selected_features.append(encoded_col_name)

X = prestamos_df[selected_features]
# Variable objetivo o variable a predecir
y = prestamos_df["debt_settlement_flag"]

In [6]:
# Dividimos el dataFrame df en df_train y df_test.
# 60% para dataset de entrenamient0, y 40% para dataset de prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11944, 10), (7964, 10), (11944,), (7964,))

In [8]:
# Creamoos el clasificador de regresion logistica
knn1 = KNeighborsClassifier(n_neighbors = 1)

# Importamos SimpleImputer para manejar valores faltantes
from sklearn.impute import SimpleImputer

# Creamos un imputador que llenará los valores NaN con la media de cada columna
imputer = SimpleImputer(strategy='mean')

# Ajustamos el imputador a los datos de entrenamiento y transformamos X_train
# Esto asegura que el imputador aprende las medias solo de los datos de entrenamiento
X_train_imputed = imputer.fit_transform(X_train)

# Transformamos X_test usando el mismo imputador. Es crucial no volver a 'fit' el imputer en X_test
X_test_imputed = imputer.transform(X_test)

# Entrenamos el clasificador
# Usamos los datos de entrenamiento imputados
knn1.fit(X_train_imputed, y_train)

# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", knn1.score(X_train_imputed, y_train) )

Precision del clasificador en fase de entrenamiento 1.0


In [10]:
# Realizar una prediccion con los datos de prueba
y_pred = knn1.predict(X_test_imputed)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador con 1 vecino: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print(f'\nMatriz Confusion con 1 vecino:\n', confusion_matrix(y_test, y_pred ))


Reporte de métricas del clasificador con 1 vecino: 
               precision    recall  f1-score   support

   No Pagado       1.00      1.00      1.00      7924
      Pagado       0.07      0.05      0.06        40

    accuracy                           0.99      7964
   macro avg       0.53      0.52      0.53      7964
weighted avg       0.99      0.99      0.99      7964


Matriz Confusion con 1 vecino:
 [[7898   26]
 [  38    2]]


In [12]:
# Creamoos el clasificador de regresion logistica
knn5 = KNeighborsClassifier(n_neighbors = 5)
# Entrenamos el clasificador
# Usamos los datos de entrenamiento imputados, ya que X_train contiene NaNs
knn5.fit(X_train_imputed, y_train)
# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", knn5.score(X_train_imputed, y_train) )

Precision del clasificador en fase de entrenamiento 0.996483590087073


In [14]:
# Realizar una prediccion con los datos de prueba
y_pred = knn5.predict(X_test_imputed)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador con 5 vecinos: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print(f'\nMatriz de Confusion con 5 vecinos:\n', confusion_matrix(y_test, y_pred ))


Reporte de métricas del clasificador con 5 vecinos: 
               precision    recall  f1-score   support

   No Pagado       0.99      1.00      1.00      7924
      Pagado       0.00      0.00      0.00        40

    accuracy                           0.99      7964
   macro avg       0.50      0.50      0.50      7964
weighted avg       0.99      0.99      0.99      7964


Matriz de Confusion con 5 vecinos:
 [[7924    0]
 [  40    0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [16]:
# Creamoos el clasificador de regresion logistica
knn10 = KNeighborsClassifier(n_neighbors = 10)
# Entrenamos el clasificador
# Usamos los datos de entrenamiento imputados
knn10.fit(X_train_imputed, y_train)
# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", knn10.score(X_train_imputed, y_train) )

Precision del clasificador en fase de entrenamiento 0.996483590087073


In [18]:
# Realizar una prediccion con los datos de prueba
y_pred = knn10.predict(X_test_imputed)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador con 10 vecinos: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print(f'Matriz de Confusion con 10 vecinos:\n', confusion_matrix(y_test, y_pred ))


Reporte de métricas del clasificador con 10 vecinos: 
               precision    recall  f1-score   support

   No Pagado       0.99      1.00      1.00      7924
      Pagado       0.00      0.00      0.00        40

    accuracy                           0.99      7964
   macro avg       0.50      0.50      0.50      7964
weighted avg       0.99      0.99      0.99      7964

Matriz de Confusion con 10 vecinos:
 [[7924    0]
 [  40    0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Balanceo de Clases con SMOTE
Dado que hemos identificado un desbalance de clases significativo en la variable objetivo, implementaremos SMOTE (Synthetic Minority Over-sampling Technique) para balancear el conjunto de datos de entrenamiento. Esto ayudará a que nuestros modelos KNN aprendan mejor de la clase minoritaria.

In [20]:
# Instalamos imblearn si no está instalado
!pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from collections import Counter

print(f"Distribución de clases antes de SMOTE: {Counter(y_train)}")

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_imputed, y_train)

print(f"Distribución de clases después de SMOTE: {Counter(y_train_res)}")

Distribución de clases antes de SMOTE: Counter({'N': 11902, 'Y': 42})
Distribución de clases después de SMOTE: Counter({'N': 11902, 'Y': 11902})


Ahora que el conjunto de datos de entrenamiento está balanceado, re-entrenaremos y evaluaremos los modelos KNN para ver si hay una mejora en la predicción de la clase minoritaria.

In [21]:
# Re-entrenamos el clasificador knn1 con datos balanceados
knn1_res = KNeighborsClassifier(n_neighbors = 1)
knn1_res.fit(X_train_res, y_train_res)

print("Precision del clasificador knn1 (resampled) en fase de entrenamiento:", knn1_res.score(X_train_res, y_train_res))

y_pred_knn1_res = knn1_res.predict(X_test_imputed)
print("\nReporte de métricas del clasificador knn1 (resampled) con 1 vecino: \n",
classification_report(y_test, y_pred_knn1_res, target_names=["No Pagado", "Pagado"]))
print(f'Matriz Confusion knn1 (resampled) con 1 vecino:\n', confusion_matrix(y_test, y_pred_knn1_res ))

Precision del clasificador knn1 (resampled) en fase de entrenamiento: 1.0

Reporte de métricas del clasificador knn1 (resampled) con 1 vecino: 
               precision    recall  f1-score   support

   No Pagado       1.00      0.92      0.96      7924
      Pagado       0.01      0.17      0.02        40

    accuracy                           0.92      7964
   macro avg       0.50      0.55      0.49      7964
weighted avg       0.99      0.92      0.95      7964

Matriz Confusion knn1 (resampled) con 1 vecino:
 [[7298  626]
 [  33    7]]


In [22]:
# Re-entrenamos el clasificador knn5 con datos balanceados
knn5_res = KNeighborsClassifier(n_neighbors = 5)
knn5_res.fit(X_train_res, y_train_res)

print("Precision del clasificador knn5 (resampled) en fase de entrenamiento:", knn5_res.score(X_train_res, y_train_res))

y_pred_knn5_res = knn5_res.predict(X_test_imputed)
print("\nReporte de métricas del clasificador knn5 (resampled) con 5 vecinos: \n",
classification_report(y_test, y_pred_knn5_res, target_names=["No Pagado", "Pagado"]))
print(f'Matriz de Confusion knn5 (resampled) con 5 vecinos:\n', confusion_matrix(y_test, y_pred_knn5_res ))

Precision del clasificador knn5 (resampled) en fase de entrenamiento: 0.9325743572508822

Reporte de métricas del clasificador knn5 (resampled) con 5 vecinos: 
               precision    recall  f1-score   support

   No Pagado       1.00      0.86      0.93      7924
      Pagado       0.01      0.17      0.01        40

    accuracy                           0.86      7964
   macro avg       0.50      0.52      0.47      7964
weighted avg       0.99      0.86      0.92      7964

Matriz de Confusion knn5 (resampled) con 5 vecinos:
 [[6852 1072]
 [  33    7]]


In [23]:
# Re-entrenamos el clasificador knn10 con datos balanceados
knn10_res = KNeighborsClassifier(n_neighbors = 10)
knn10_res.fit(X_train_res, y_train_res)

print("Precision del clasificador knn10 (resampled) en fase de entrenamiento:", knn10_res.score(X_train_res, y_train_res))

y_pred_knn10_res = knn10_res.predict(X_test_imputed)
print("\nReporte de métricas del clasificador knn10 (resampled) con 10 vecinos: \n",
classification_report(y_test, y_pred_knn10_res, target_names=["No Pagado", "Pagado"]))
print(f'Matriz de Confusion knn10 (resampled) con 10 vecinos:\n', confusion_matrix(y_test, y_pred_knn10_res ))

Precision del clasificador knn10 (resampled) en fase de entrenamiento: 0.8980003360779701

Reporte de métricas del clasificador knn10 (resampled) con 10 vecinos: 
               precision    recall  f1-score   support

   No Pagado       1.00      0.85      0.92      7924
      Pagado       0.01      0.20      0.01        40

    accuracy                           0.85      7964
   macro avg       0.50      0.53      0.47      7964
weighted avg       0.99      0.85      0.91      7964

Matriz de Confusion knn10 (resampled) con 10 vecinos:
 [[6748 1176]
 [  32    8]]


### Exploración de Importancia de Características con un Modelo más Robusto (LightGBM)

Dado que el modelo KNN mostró una baja efectividad en la clase minoritaria y la importancia de características resultó en valores cercanos a cero, exploraremos un modelo más robusto, como el **LightGBM Classifier**. Este tipo de modelo de ensamble basado en árboles suele manejar mejor los datos desbalanceados y proporciona una evaluación de importancia de características más significativa.

In [27]:
# Instalar LightGBM si no está instalado
!pip install lightgbm

import lightgbm as lgb

# Creamos el clasificador LightGBM
# Usamos 'is_unbalance=True' para ayudar a manejar el desbalance de clases si no se usara SMOTE,
# pero con SMOTE, el dataset de entrenamiento ya está balanceado.
# Se pueden ajustar otros parámetros como 'scale_pos_weight' si el desbalance persiste o si no se usa SMOTE.
lgbm_clf = lgb.LGBMClassifier(objective='binary', random_state=42, n_estimators=500, learning_rate=0.05, num_leaves=31)

# Entrenamos el clasificador con los datos balanceados (SMOTE)
lgbm_clf.fit(X_train_res, y_train_res)

# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador LightGBM en fase de entrenamiento:", lgbm_clf.score(X_train_res, y_train_res))

[LightGBM] [Info] Number of positive: 11902, number of negative: 11902
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003369 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 23804, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



Precision del clasificador LightGBM en fase de entrenamiento: 1.0


In [28]:
# Realizar una prediccion con los datos de prueba
y_pred_lgbm = lgbm_clf.predict(X_test_imputed)

# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador LightGBM: \n",
classification_report(y_test, y_pred_lgbm, target_names=["No Pagado", "Pagado"]))
print(f'Matriz de Confusion LightGBM:\n', confusion_matrix(y_test, y_pred_lgbm ))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names




Reporte de métricas del clasificador LightGBM: 
               precision    recall  f1-score   support

   No Pagado       0.99      1.00      1.00      7924
      Pagado       0.00      0.00      0.00        40

    accuracy                           0.99      7964
   macro avg       0.50      0.50      0.50      7964
weighted avg       0.99      0.99      0.99      7964

Matriz de Confusion LightGBM:
 [[7923    1]
 [  40    0]]


In [29]:
# Obtener la importancia de las características del modelo LightGBM
feature_importance = pd.DataFrame({
    'Variable': X.columns,
    'Importancia Media': lgbm_clf.feature_importances_
}).sort_values('Importancia Media', ascending=False)

print("\nImportancia de las características (LightGBM):\n")
print(feature_importance)


Importancia de las características (LightGBM):

               Variable  Importancia Media
2            annual_inc               2500
6            grade_code               1871
3                   dti               1864
0           funded_amnt               1863
4            revol_util               1795
8       addr_state_code               1435
1              int_rate               1434
7          purpose_code               1310
9   home_ownership_code                773
5  pub_rec_bankruptcies                155


In [30]:
# Visualizar la importancia de las características con Plotly
import plotly.express as px

fig_lgbm_importance = px.bar(feature_importance, x="Variable", y="Importancia Media",
                             title="Importancia de Características (LightGBM)",
                             text_auto=".3f", color="Variable")
fig_lgbm_importance.update_layout(width=900, height=600)
fig_lgbm_importance.show()

### LightGBM con `scale_pos_weight` (sin SMOTE)

In [32]:
from sklearn.utils import class_weight

# Calculamos los pesos de las clases del dataset de entrenamiento original
# Esto es importante para que el modelo ponga más énfasis en la clase minoritaria
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convertir a diccionario para facilitar el acceso
class_weight_dict = dict(zip(np.unique(y_train), class_weights))

# El parámetro 'scale_pos_weight' en LightGBM se refiere al ratio de la clase negativa a la positiva.
# Si 'N' es la mayoría y 'Y' es la minoría, queremos aumentar el peso de 'Y'.
# LightGBM espera el ratio (número de muestras de la clase mayoritaria / número de muestras de la clase minoritaria)
# En nuestro caso, la clase mayoritaria es 'N' y la minoritaria es 'Y'.
# y_train = prestamos_df["debt_settlement_flag"]

count_negative = (y_train == 'N').sum()
count_positive = (y_train == 'Y').sum()

scale_pos_weight_value = count_negative / count_positive

print(f"Número de muestras 'N' (mayoritaria): {count_negative}")
print(f"Número de muestras 'Y' (minoritaria): {count_positive}")
print(f"scale_pos_weight calculado: {scale_pos_weight_value:.2f}")

# Creamos el clasificador LightGBM con scale_pos_weight
# Usamos X_train_imputed (sin SMOTE) y el y_train original
lgbm_clf_weighted = lgb.LGBMClassifier(
    objective='binary',
    random_state=42,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight_value # Aplicamos el peso para la clase positiva
)

# Entrenamos el clasificador con los datos de entrenamiento imputados (sin SMOTE)
lgbm_clf_weighted.fit(X_train_imputed, y_train)

# Precisión del modelo en la fase de entrenamiento
print("\nPrecision del clasificador LightGBM (weighted) en fase de entrenamiento:", lgbm_clf_weighted.score(X_train_imputed, y_train))

Número de muestras 'N' (mayoritaria): 11902
Número de muestras 'Y' (minoritaria): 42
scale_pos_weight calculado: 283.38
[LightGBM] [Info] Number of positive: 42, number of negative: 11902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001068 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1274
[LightGBM] [Info] Number of data points in the train set: 11944, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.003516 -> initscore=-5.646792
[LightGBM] [Info] Start training from score -5.646792
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names




Precision del clasificador LightGBM (weighted) en fase de entrenamiento: 1.0


In [33]:
# Realizar una prediccion con los datos de prueba usando el modelo ponderado
y_pred_lgbm_weighted = lgbm_clf_weighted.predict(X_test_imputed)

# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador LightGBM (weighted): \n",
classification_report(y_test, y_pred_lgbm_weighted, target_names=["No Pagado", "Pagado"]))
print(f'Matriz de Confusion LightGBM (weighted):\n', confusion_matrix(y_test, y_pred_lgbm_weighted ))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names




Reporte de métricas del clasificador LightGBM (weighted): 
               precision    recall  f1-score   support

   No Pagado       0.99      1.00      1.00      7924
      Pagado       0.00      0.00      0.00        40

    accuracy                           0.99      7964
   macro avg       0.50      0.50      0.50      7964
weighted avg       0.99      0.99      0.99      7964

Matriz de Confusion LightGBM (weighted):
 [[7920    4]
 [  40    0]]


In [35]:
from sklearn.preprocessing import StandardScaler

# Inicializamos el StandardScaler
scaler = StandardScaler()

# Ajustamos el scaler a los datos de entrenamiento imputados y los transformamos
X_train_scaled = scaler.fit_transform(X_train_imputed)

# Transformamos los datos de prueba imputados usando el scaler ya ajustado
X_test_scaled = scaler.transform(X_test_imputed)

In [36]:
# Entrenar KNN
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled)
print("\nReporte de clasificación datos normalizados y 5 vecinos:")
print(classification_report(y_test, y_pred_scaled, target_names=["No Pagado", "Pagado"]))
print ("\nMatriz de confusión con datos normalizados y 5 vecinos:\n", confusion_matrix(y_test, y_pred_scaled), "\n")


Reporte de clasificación datos normalizados y 5 vecinos:
              precision    recall  f1-score   support

   No Pagado       0.99      1.00      1.00      7924
      Pagado       0.00      0.00      0.00        40

    accuracy                           0.99      7964
   macro avg       0.50      0.50      0.50      7964
weighted avg       0.99      0.99      0.99      7964


Matriz de confusión con datos normalizados y 5 vecinos:
 [[7924    0]
 [  40    0]] 



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [37]:
# Entrenar KNN
knn_scaled = KNeighborsClassifier(n_neighbors=10)
knn_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled)
print("\nReporte de clasificación datos normalizados y 10 vecinos:")
print(classification_report(y_test, y_pred_scaled, target_names=["No Pagado", "Pagado"]))
print ("\nMatriz de confusión con datos normalizados y 10 vecinos:\n", confusion_matrix(y_test, y_pred_scaled), "\n")


Reporte de clasificación datos normalizados y 10 vecinos:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



              precision    recall  f1-score   support

   No Pagado       0.99      1.00      1.00      7924
      Pagado       0.00      0.00      0.00        40

    accuracy                           0.99      7964
   macro avg       0.50      0.50      0.50      7964
weighted avg       0.99      0.99      0.99      7964


Matriz de confusión con datos normalizados y 10 vecinos:
 [[7924    0]
 [  40    0]] 



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

